# Meta-PhyGT DNR Single-File Notebook

Meta-Physical Graph Transformer (Meta-PhyGT) for IEEE 33-bus DNR.

Single-file implementation:
  - loads dnr_diagnostic.h5 generated by 01_DNR_Data_Generation.ipynb
  - builds a PyTorch Geometric graph dataset
  - trains Meta-PhyGT with Reptile meta-learning
  - validates projected radial topologies with pandapower AC power flow
  - writes plots/metrics to disk

No checkpoint loading or external model imports are used.

In [ ]:
from __future__ import annotations

import argparse
import ast
import math
import random
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import h5py
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pandapower as pp
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch


# ═══════════════════════════════════════════════════════════════════

## CHAPTER 6: RUN EXECUTION & HYPERPARAMETER TUNING

In [ ]:
# Tunable globals live at the top by design.
inner_lr = 1e-3
meta_lr = 1e-4
inner_steps = 5
epochs = 50
w_bce = 1.0
w_mse = 0.5
w_volt = 2.0

support_size = 5
query_size = 16
validation_batch_size = 64
validation_interval = 5
critic_cases_per_eval = 16
random_seed = 42

hidden_dim = 96
num_heads = 4
num_layers = 3
structural_dim = 6
dropout = 0.10

DIAG_FILE = "dnr_diagnostic.h5"
V_MIN_PU = 0.90
VN_KV = 12.66
BASE_MVA = 10.0
N_BUS = 33
N_LINES = 37
N_OPEN = 5


# IEEE 33-bus full DNR action graph: 32 section lines + 5 tie lines.
FROM_BUS = np.array(
    [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,
        2, 19, 20, 21, 3, 23, 24, 6, 26, 27, 28, 29, 30, 31, 32,
        21, 9, 12, 18, 25,
    ],
    dtype=np.int64,
) - 1
TO_BUS = np.array(
    [
        2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        8, 15, 22, 33, 29,
    ],
    dtype=np.int64,
) - 1
R_OHM = np.array(
    [
        0.0922, 0.4930, 0.3660, 0.3811, 0.8190, 0.1872, 1.7114, 1.0300, 1.0440,
        0.1966, 0.3744, 1.4680, 0.5416, 0.5910, 0.7463, 1.2890, 0.7320,
        0.1640, 1.5042, 0.4095, 0.7089, 0.4512, 0.8980, 0.8960, 0.2030, 0.2842,
        1.0590, 0.8042, 0.5075, 0.9744, 0.3105, 0.3410, 2.0000, 2.0000, 2.0000,
        0.5000, 0.5000,
    ],
    dtype=np.float64,
)
X_OHM = np.array(
    [
        0.0470, 0.2511, 0.1864, 0.1941, 0.7070, 0.6188, 1.2351, 0.7400, 0.7400,
        0.0650, 0.1238, 1.1550, 0.7129, 0.5260, 0.5450, 1.7210, 0.5740,
        0.1565, 1.3554, 0.4784, 0.9373, 0.3083, 0.7091, 0.7011, 0.1034, 0.1447,
        0.9337, 0.7006, 0.2585, 0.9630, 0.3619, 0.5302, 2.0000, 2.0000, 2.0000,
        0.5000, 0.5000,
    ],
    dtype=np.float64,
)


@dataclass
class RunConfig:
    h5_path: str = DIAG_FILE
    epochs: int = epochs
    inner_steps: int = inner_steps
    inner_lr: float = inner_lr
    meta_lr: float = meta_lr
    support_size: int = support_size
    query_size: int = query_size
    critic_cases: int = critic_cases_per_eval
    validation_interval: int = validation_interval
    seed: int = random_seed
    device: str = "cuda"
    output_dir: str = "metaphygt_outputs"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## CHAPTER 1: DATA PIPELINE & PYG GRAPH DATASET CONSTRUCTION

In [ ]:
def inspect_hdf5(path: str) -> None:
    print("HDF5 structure:")
    with h5py.File(path, "r") as h5:
        def visit(name, obj):
            print(f"  {name:42s} {type(obj).__name__:18s} {getattr(obj, 'shape', '')}")

        h5.visititems(visit)


def load_hdf_tables(path: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not Path(path).exists():
        raise FileNotFoundError(f"{path} not found. Run 01_DNR_Data_Generation.ipynb first.")
    inspect_hdf5(path)
    with pd.HDFStore(path, mode="r") as store:
        keys = {k.strip("/") for k in store.keys()}
        if "scenarios" not in keys:
            raise KeyError(f"{path} missing required key 'scenarios'. Keys={sorted(keys)}")
        scenarios = store["scenarios"]
        summary = store["summary"] if "summary" in keys else pd.DataFrame()
        matrix_check = store["training_matrix_check"] if "training_matrix_check" in keys else pd.DataFrame()
        loss_weights = store["loss_weights"] if "loss_weights" in keys else pd.DataFrame()
    print(f"Loaded scenarios: {scenarios.shape}")
    print(f"Available tables: scenarios, summary={not summary.empty}, "
          f"training_matrix_check={not matrix_check.empty}, loss_weights={not loss_weights.empty}")
    return scenarios, summary, matrix_check, loss_weights


def make_edge_tensors(device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    edge_index = torch.as_tensor(np.stack([FROM_BUS, TO_BUS]), dtype=torch.long, device=device)
    tie_flag = (np.arange(N_LINES) >= (N_LINES - N_OPEN)).astype(np.float32)
    line_norm = np.arange(N_LINES, dtype=np.float32) / (N_LINES - 1)
    edge_attr = np.stack(
        [
            R_OHM / (R_OHM.max() + 1e-9),
            X_OHM / (X_OHM.max() + 1e-9),
            tie_flag,
            line_norm,
        ],
        axis=1,
    )
    return edge_index, torch.as_tensor(edge_attr, dtype=torch.float32, device=device)


def infer_task(row: pd.Series) -> str:
    season = str(row.get("season", ""))
    if season == "steered":
        return "B_topology_steered"
    if season == "stress":
        return "C_stress_sampler"
    return "A_realistic"


def parse_topology(value) -> np.ndarray:
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.array(value, dtype=np.int64)
    return np.array(ast.literal_eval(str(value)), dtype=np.int64)


class DNRMetaDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, edge_index: torch.Tensor, edge_attr: torch.Tensor):
        super().__init__()
        self.frame = frame.reset_index(drop=True).copy()
        self.edge_index_cpu = edge_index.detach().cpu()
        self.edge_attr_cpu = edge_attr.detach().cpu()
        self.p_cols = [f"p_bus_{b}" for b in range(1, N_BUS)]
        self.q_cols = [f"q_bus_{b}" for b in range(1, N_BUS)]
        missing = [c for c in self.p_cols + self.q_cols + ["topo", "loss_mw", "v_min_pu"]
                   if c not in self.frame.columns]
        if missing:
            raise KeyError(f"Missing required scenario columns: {missing}")

        self.p_scale = max(float(np.nanmax(self.frame[self.p_cols].to_numpy(dtype=np.float32))), 1e-6)
        self.q_scale = max(float(np.nanmax(self.frame[self.q_cols].to_numpy(dtype=np.float32))), 1e-6)
        self.loss_mu = float(self.frame["loss_mw"].mean())
        self.loss_sigma = float(self.frame["loss_mw"].std() + 1e-6)
        self.vmin_mu = float(self.frame["v_min_pu"].mean())
        self.vmin_sigma = float(self.frame["v_min_pu"].std() + 1e-6)

        self.season_vocab = ["winter", "summer", "spring", "autumn", "steered", "stress"]
        self.season_to_idx = {s: i for i, s in enumerate(self.season_vocab)}
        self.bus_id_norm = np.arange(N_BUS, dtype=np.float32) / (N_BUS - 1)
        self.slack_flag = (np.arange(N_BUS) == 0).astype(np.float32)
        self.task_labels = [infer_task(row) for _, row in self.frame.iterrows()]
        self.task_to_indices: Dict[str, List[int]] = defaultdict(list)
        for idx, task in enumerate(self.task_labels):
            self.task_to_indices[task].append(idx)

    def len(self) -> int:
        return len(self.frame)

    def _load_vectors(self, row: pd.Series) -> Tuple[np.ndarray, np.ndarray]:
        p = np.zeros(N_BUS, dtype=np.float32)
        q = np.zeros(N_BUS, dtype=np.float32)
        p[1:] = row[self.p_cols].to_numpy(dtype=np.float32)
        q[1:] = row[self.q_cols].to_numpy(dtype=np.float32)
        return p, q

    def _node_features(self, row: pd.Series) -> np.ndarray:
        p, q = self._load_vectors(row)
        hour = float(row.get("hour", -1))
        hour_norm = 0.0 if hour < 0 else hour / 23.0
        flags = np.tile(
            np.array(
                [
                    hour_norm,
                    1.0 if hour < 0 else 0.0,
                    float(bool(row.get("weekend", False))),
                    float(bool(row.get("ev_active", False))),
                    float(bool(row.get("pv_active", False))),
                    float(bool(row.get("default_voltage_violation", False))),
                    float(bool(row.get("recovered_voltage_violation", False))),
                ],
                dtype=np.float32,
            ),
            (N_BUS, 1),
        )
        season_onehot = np.zeros(len(self.season_vocab), dtype=np.float32)
        season = str(row.get("season", ""))
        if season in self.season_to_idx:
            season_onehot[self.season_to_idx[season]] = 1.0
        season_rep = np.tile(season_onehot, (N_BUS, 1))
        return np.column_stack(
            [
                p / self.p_scale,
                q / self.q_scale,
                self.bus_id_norm,
                self.slack_flag,
                flags,
                season_rep,
            ]
        ).astype(np.float32)

    def get(self, idx: int) -> Data:
        row = self.frame.iloc[int(idx)]
        y = torch.zeros(N_LINES, dtype=torch.float32)
        y[torch.as_tensor(parse_topology(row["topo"]) - 1, dtype=torch.long)] = 1.0
        p, q = self._load_vectors(row)
        loss = float(row["loss_mw"])
        vmin = float(row["v_min_pu"])
        return Data(
            x=torch.as_tensor(self._node_features(row), dtype=torch.float32),
            edge_index=self.edge_index_cpu.clone(),
            edge_attr=self.edge_attr_cpu.clone(),
            y=y,
            loss_target=torch.tensor([loss], dtype=torch.float32),
            loss_norm_target=torch.tensor([(loss - self.loss_mu) / self.loss_sigma], dtype=torch.float32),
            vmin_target=torch.tensor([vmin], dtype=torch.float32),
            p_mw=torch.as_tensor(p, dtype=torch.float32),
            q_mvar=torch.as_tensor(q, dtype=torch.float32),
            scenario_idx=torch.tensor([idx], dtype=torch.long),
        )

## CHAPTER 2: PHYSICS-GATED GRAPH TRANSFORMER ARCHITECTURE

In [ ]:
def structural_eigenvectors(dim: int) -> np.ndarray:
    adj = np.zeros((N_BUS, N_BUS), dtype=np.float32)
    for i, j in zip(FROM_BUS, TO_BUS):
        adj[i, j] = 1.0
        adj[j, i] = 1.0
    lap = np.diag(adj.sum(axis=1)) - adj
    _, eigvecs = np.linalg.eigh(lap)
    return eigvecs[:, 1:dim + 1].astype(np.float32)


def impedance_distance_matrix(device: torch.device) -> torch.Tensor:
    graph = nx.Graph()
    for k, (i, j) in enumerate(zip(FROM_BUS, TO_BUS)):
        graph.add_edge(int(i), int(j), weight=float(math.hypot(R_OHM[k], X_OHM[k])))
    dist = np.zeros((N_BUS, N_BUS), dtype=np.float32)
    for i in range(N_BUS):
        lengths = nx.single_source_dijkstra_path_length(graph, i, weight="weight")
        for j in range(N_BUS):
            dist[i, j] = lengths.get(j, 0.0)
    dist /= max(float(dist.max()), 1e-6)
    return torch.as_tensor(dist, dtype=torch.float32, device=device)


class StructuralEncoding(nn.Module):
    def __init__(self, eig_features: np.ndarray):
        super().__init__()
        self.register_buffer("eig_features", torch.as_tensor(eig_features, dtype=torch.float32))

    def forward(self, x: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        dense_x, mask = to_dense_batch(x, batch)
        bsz = dense_x.shape[0]
        eig = self.eig_features.to(x.device).unsqueeze(0).expand(bsz, -1, -1)
        return torch.cat([dense_x, eig], dim=-1)[mask]


class ImpedanceGatedAttention(nn.Module):
    def __init__(self, dim: int, heads: int, dropout_p: float):
        super().__init__()
        if dim % heads != 0:
            raise ValueError("dim must be divisible by heads")
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.gamma = nn.Parameter(torch.tensor(0.10))
        self.dropout = nn.Dropout(dropout_p)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x: torch.Tensor, batch: torch.Tensor, z_matrix: torch.Tensor) -> torch.Tensor:
        dense_x, mask = to_dense_batch(x, batch)
        bsz, nodes, dim = dense_x.shape
        qkv = self.qkv(dense_x).view(bsz, nodes, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores - F.softplus(self.gamma) * z_matrix[:nodes, :nodes].view(1, 1, nodes, nodes)
        scores = scores.masked_fill(~mask.view(bsz, 1, 1, nodes), -1e9)
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bsz, nodes, dim)
        h = self.norm1(dense_x + self.proj(out))
        h = self.norm2(h + self.ff(h))
        return h[mask]


class MetaPhyGT(nn.Module):
    def __init__(
        self,
        node_dim: int,
        device: torch.device,
        hidden: int = hidden_dim,
        heads: int = num_heads,
        layers: int = num_layers,
        struct_dim: int = structural_dim,
        dropout_p: float = dropout,
    ):
        super().__init__()
        self.structural = StructuralEncoding(structural_eigenvectors(struct_dim))
        self.z_matrix = impedance_distance_matrix(device)
        self.input = nn.Linear(node_dim + struct_dim, hidden)
        self.layers = nn.ModuleList([ImpedanceGatedAttention(hidden, heads, dropout_p) for _ in range(layers)])
        self.edge_head = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 1),
        )
        self.physics_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 2),
        )

    def forward(self, data: Data) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        h = self.structural(data.x, data.batch)
        h = self.input(h)
        for layer in self.layers:
            h = layer(h, data.batch, self.z_matrix)
        dense_h, mask = to_dense_batch(h, data.batch)
        src = torch.as_tensor(FROM_BUS, dtype=torch.long, device=h.device)
        dst = torch.as_tensor(TO_BUS, dtype=torch.long, device=h.device)
        edge_logits = self.edge_head(torch.cat([dense_h[:, src, :], dense_h[:, dst, :]], dim=-1)).squeeze(-1)
        pooled = (dense_h * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp_min(1)
        physics = self.physics_head(pooled)
        pred_loss_norm = physics[:, 0]
        pred_vmin = 0.75 + 0.35 * torch.sigmoid(physics[:, 1])
        return edge_logits, pred_loss_norm, pred_vmin

## CHAPTER 3: REPTILE + PHYSICS-INFORMED LOSS

In [ ]:
def make_splits(dataset: DNRMetaDataset, val_fraction: float = 0.20) -> Tuple[List[int], List[int]]:
    indices = np.arange(len(dataset))
    rng = np.random.default_rng(random_seed)
    rng.shuffle(indices)
    val_n = max(1, int(len(indices) * val_fraction))
    return indices[val_n:].tolist(), indices[:val_n].tolist()


def make_loader(dataset: DNRMetaDataset, indices: Iterable[int], batch: int, shuffle: bool) -> DataLoader:
    return DataLoader([dataset[i] for i in indices], batch_size=batch, shuffle=shuffle)


def positive_line_weights(dataset: DNRMetaDataset, train_idx: List[int], device: torch.device) -> torch.Tensor:
    y = torch.stack([dataset[i].y for i in train_idx])
    pos = y.sum(dim=0)
    neg = len(train_idx) - pos
    return (neg / pos.clamp_min(1.0)).clamp(1.0, 50.0).to(device)


def multi_objective_loss(
    model: MetaPhyGT,
    batch: Data,
    pos_weight: torch.Tensor,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    logits, pred_loss_norm, pred_vmin = model(batch)
    target_edges = batch.y.view(-1, N_LINES)
    target_loss_norm = batch.loss_norm_target.view(-1)
    bce = F.binary_cross_entropy_with_logits(logits, target_edges, pos_weight=pos_weight)
    mse = F.mse_loss(pred_loss_norm, target_loss_norm)
    volt = torch.relu(V_MIN_PU - pred_vmin).pow(2).mean()
    total = w_bce * bce + w_mse * mse + w_volt * volt
    return total, {"bce": float(bce.detach().cpu()), "mse": float(mse.detach().cpu()), "volt": float(volt.detach().cpu())}


def evaluate_loss(
    model: MetaPhyGT,
    dataset: DNRMetaDataset,
    indices: List[int],
    pos_weight: torch.Tensor,
    device: torch.device,
) -> float:
    model.eval()
    losses: List[float] = []
    with torch.no_grad():
        for batch in make_loader(dataset, indices, validation_batch_size, shuffle=False):
            batch = batch.to(device)
            loss, _ = multi_objective_loss(model, batch, pos_weight)
            losses.append(float(loss.cpu()))
    return float(np.mean(losses)) if losses else float("nan")


def reptile_train(
    model: MetaPhyGT,
    dataset: DNRMetaDataset,
    train_idx: List[int],
    val_idx: List[int],
    pos_weight: torch.Tensor,
    cfg: RunConfig,
    device: torch.device,
) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.seed)
    task_to_train = defaultdict(list)
    for idx in train_idx:
        task_to_train[dataset.task_labels[idx]].append(idx)
    tasks = [v for v in task_to_train.values() if v]
    history = []
    for epoch in range(1, cfg.epochs + 1):
        task = tasks[int(rng.integers(0, len(tasks)))]
        support = rng.choice(task, size=min(cfg.support_size, len(task)), replace=len(task) < cfg.support_size)
        fast = copy_model(model, device)
        opt = torch.optim.Adam(fast.parameters(), lr=cfg.inner_lr)
        for _ in range(cfg.inner_steps):
            for batch in make_loader(dataset, support.tolist(), len(support), shuffle=True):
                batch = batch.to(device)
                opt.zero_grad()
                loss, _ = multi_objective_loss(fast, batch, pos_weight)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(fast.parameters(), 2.0)
                opt.step()
        with torch.no_grad():
            for p, fp in zip(model.parameters(), fast.parameters()):
                p.add_(cfg.meta_lr * (fp - p))
        if epoch == 1 or epoch % cfg.validation_interval == 0 or epoch == cfg.epochs:
            tr = evaluate_loss(model, dataset, train_idx[: min(len(train_idx), validation_batch_size)], pos_weight, device)
            vl = evaluate_loss(model, dataset, val_idx, pos_weight, device)
            history.append({"epoch": epoch, "meta_train_loss": tr, "meta_val_loss": vl})
            print(f"Epoch {epoch:03d}/{cfg.epochs}: meta_train={tr:.5f} meta_val={vl:.5f}")
    return pd.DataFrame(history)


def copy_model(model: MetaPhyGT, device: torch.device) -> MetaPhyGT:
    clone = MetaPhyGT(model.input.in_features - structural_dim, device).to(device)
    clone.load_state_dict(model.state_dict())
    return clone

## CHAPTER 4: PHYSICAL OFFSET ANALYSIS & PANDAPOWER COMPARATIVE CRITIC

In [ ]:
class PhysicalValidationCritic:
    def __init__(self):
        self.net = self._build_network()

    @staticmethod
    def _build_network():
        net = pp.create_empty_network(sn_mva=BASE_MVA)
        for _ in range(N_BUS):
            pp.create_bus(net, vn_kv=VN_KV)
        pp.create_ext_grid(net, bus=0, vm_pu=1.0)
        for bus in range(1, N_BUS):
            pp.create_load(net, bus=bus, p_mw=0.0, q_mvar=0.0)
        for fb, tb, r, x in zip(FROM_BUS, TO_BUS, R_OHM, X_OHM):
            pp.create_line_from_parameters(net, int(fb), int(tb), 1.0, float(r), float(x), 0.0, 1.0)
        return net

    @staticmethod
    def radial_projection(open_logits: torch.Tensor) -> np.ndarray:
        open_prob = torch.sigmoid(open_logits.detach()).cpu().numpy()
        graph = nx.Graph()
        graph.add_nodes_from(range(N_BUS))
        for k, (i, j) in enumerate(zip(FROM_BUS, TO_BUS)):
            graph.add_edge(int(i), int(j), line_idx=k, weight=float(1.0 - open_prob[k]))
        tree = nx.maximum_spanning_tree(graph, weight="weight")
        active = {data["line_idx"] for _, _, data in tree.edges(data=True)}
        open_idx = sorted(set(range(N_LINES)) - active)
        if len(open_idx) != N_OPEN:
            open_idx = np.argsort(-open_prob)[:N_OPEN].tolist()
        return np.array(open_idx, dtype=np.int64)

    def runpp(self, p: np.ndarray, q: np.ndarray, open_idx: np.ndarray) -> Dict[str, float]:
        for load_idx, bus in enumerate(range(1, N_BUS)):
            self.net.load.at[load_idx, "p_mw"] = float(p[bus])
            self.net.load.at[load_idx, "q_mvar"] = float(q[bus])
        self.net.line.loc[:, "in_service"] = True
        self.net.line.loc[open_idx, "in_service"] = False
        for algorithm in ["bfsw", "nr", "iwamoto_nr"]:
            try:
                pp.runpp(
                    self.net,
                    algorithm=algorithm,
                    init="flat",
                    tolerance_mva=1e-9,
                    max_iteration=200,
                    calculate_voltage_angles=False,
                    numba=False,
                )
                if bool(self.net.converged):
                    return {
                        "converged": True,
                        "loss_mw": float(self.net.res_line.pl_mw.sum()),
                        "vmin_pu": float(self.net.res_bus.vm_pu.min()),
                        "voltage_violation": bool(self.net.res_bus.vm_pu.min() < V_MIN_PU),
                        "algorithm": algorithm,
                    }
            except Exception:
                continue
        return {"converged": False, "loss_mw": np.nan, "vmin_pu": np.nan, "voltage_violation": True, "algorithm": None}

    def evaluate(self, model: MetaPhyGT, dataset: DNRMetaDataset, indices: List[int], device: torch.device) -> pd.DataFrame:
        model.eval()
        rows = []
        with torch.no_grad():
            for batch in make_loader(dataset, indices, 1, shuffle=False):
                batch = batch.to(device)
                logits, pred_loss_norm, pred_vmin = model(batch)
                open_idx = self.radial_projection(logits[0])
                p = batch.p_mw.view(-1, N_BUS)[0].detach().cpu().numpy()
                q = batch.q_mvar.view(-1, N_BUS)[0].detach().cpu().numpy()
                pp_res = self.runpp(p, q, open_idx)
                pred_loss = float((pred_loss_norm[0] * dataset.loss_sigma + dataset.loss_mu).detach().cpu())
                rows.append(
                    {
                        "scenario_idx": int(batch.scenario_idx.view(-1)[0].detach().cpu()),
                        "pred_open_lines": tuple((open_idx + 1).tolist()),
                        "pred_loss_mw": pred_loss,
                        "pred_vmin_pu": float(pred_vmin[0].detach().cpu()),
                        "pp_loss_mw": pp_res["loss_mw"],
                        "pp_vmin_pu": pp_res["vmin_pu"],
                        "delta_loss": abs(pred_loss - pp_res["loss_mw"]) if pp_res["converged"] else np.nan,
                        "delta_voltage_offset": abs(float(pred_vmin[0].detach().cpu()) - pp_res["vmin_pu"]) if pp_res["converged"] else np.nan,
                        "pp_voltage_violation": pp_res["voltage_violation"],
                        "pp_converged": pp_res["converged"],
                        "pp_algorithm": pp_res["algorithm"],
                    }
                )
        return pd.DataFrame(rows)

## CHAPTER 5: REAL-TIME CONVERGENCE & ANALYSIS VISUALIZATION

In [ ]:
def plot_results(history: pd.DataFrame, critic: pd.DataFrame, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    if not history.empty:
        axes[0].plot(history["epoch"], history["meta_train_loss"], marker="o", label="Meta-train")
        axes[0].plot(history["epoch"], history["meta_val_loss"], marker="s", label="Meta-val")
    axes[0].set_title("Loss Convergence Tracking")
    axes[0].legend()

    offsets = critic[["delta_loss", "delta_voltage_offset"]].melt(var_name="metric", value_name="offset").dropna()
    if not offsets.empty:
        sns.violinplot(data=offsets, x="metric", y="offset", ax=axes[1], inner="quartile")
    axes[1].set_title("Physical Error Metric Distribution")

    if "pred_open_lines" in critic and len(critic) > 0:
        tie_lines = [33, 34, 35, 36, 37]
        mat = np.zeros((len(critic), len(tie_lines)), dtype=float)
        for r, lines in enumerate(critic["pred_open_lines"]):
            mat[r] = [1.0 if line in lines else 0.0 for line in tie_lines]
        sns.heatmap(mat, ax=axes[2], cmap="viridis", cbar=True, xticklabels=tie_lines)
    axes[2].set_title("Projected Tie-Switch Open Decisions")
    axes[2].set_xlabel("Tie line")
    axes[2].set_ylabel("Critic case")

    fig.tight_layout()
    fig.savefig(output_dir / "metaphygt_training_and_critic.png", dpi=180)
    plt.close(fig)
    print(f"Saved plot: {output_dir / 'metaphygt_training_and_critic.png'}")


def print_summary(dataset: DNRMetaDataset, history: pd.DataFrame, critic: pd.DataFrame) -> None:
    print("═" * 80)
    print("Meta-PhyGT DNR experiment complete")
    print("═" * 80)
    print(f"Dataset scenarios      : {len(dataset):,}")
    print(f"Task groups            : {dict((k, len(v)) for k, v in dataset.task_to_indices.items())}")
    if not history.empty:
        print(f"Final meta-train loss  : {history.meta_train_loss.iloc[-1]:.5f}")
        print(f"Final meta-val loss    : {history.meta_val_loss.iloc[-1]:.5f}")
    if not critic.empty:
        print(f"Critic cases           : {len(critic)}")
        print(f"Pandapower convergence : {critic.pp_converged.mean() * 100:.1f}%")
        print(f"Mean ΔLoss             : {critic.delta_loss.mean():.6f} MW")
        print(f"Mean ΔVoltage Offset   : {critic.delta_voltage_offset.mean():.6f} pu")
        print(f"Voltage violations     : {int(critic.pp_voltage_violation.sum())}")
    print("═" * 80)

## CHAPTER 6: MAIN ORCHESTRATOR

In [ ]:
def main(cfg: RunConfig) -> Dict[str, object]:
    seed_everything(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() and cfg.device == "cuda" else "cpu")
    print(f"Using device: {device}")

    scenarios, summary, matrix_check, loss_weights = load_hdf_tables(cfg.h5_path)
    edge_index, edge_attr = make_edge_tensors(device)
    dataset = DNRMetaDataset(scenarios, edge_index, edge_attr)
    print(f"Dataset size: {len(dataset)}")
    print(f"Node feature shape: {tuple(dataset[0].x.shape)}")
    print(f"Tasks: {dict((k, len(v)) for k, v in dataset.task_to_indices.items())}")

    train_idx, val_idx = make_splits(dataset)
    model = MetaPhyGT(dataset[0].x.shape[1], device).to(device)
    pos_weight = positive_line_weights(dataset, train_idx, device)
    history = reptile_train(model, dataset, train_idx, val_idx, pos_weight, cfg, device)

    critic = PhysicalValidationCritic()
    critic_idx = val_idx[: min(cfg.critic_cases, len(val_idx))]
    critic_df = critic.evaluate(model, dataset, critic_idx, device)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    history.to_csv(output_dir / "metaphygt_history.csv", index=False)
    critic_df.to_csv(output_dir / "metaphygt_physical_critic.csv", index=False)
    plot_results(history, critic_df, output_dir)
    print_summary(dataset, history, critic_df)
    return {"model": model, "dataset": dataset, "history": history, "critic": critic_df}


def parse_args() -> RunConfig:
    parser = argparse.ArgumentParser(description="Single-file Meta-PhyGT DNR training and validation.")
    parser.add_argument("--h5", default=DIAG_FILE)
    parser.add_argument("--epochs", type=int, default=epochs)
    parser.add_argument("--inner-steps", type=int, default=inner_steps)
    parser.add_argument("--inner-lr", type=float, default=inner_lr)
    parser.add_argument("--meta-lr", type=float, default=meta_lr)
    parser.add_argument("--support-size", type=int, default=support_size)
    parser.add_argument("--query-size", type=int, default=query_size)
    parser.add_argument("--critic-cases", type=int, default=critic_cases_per_eval)
    parser.add_argument("--validation-interval", type=int, default=validation_interval)
    parser.add_argument("--seed", type=int, default=random_seed)
    parser.add_argument("--device", default="cuda", choices=["cuda", "cpu"])
    parser.add_argument("--output-dir", default="metaphygt_outputs")
    args = parser.parse_args()
    return RunConfig(
        h5_path=args.h5,
        epochs=args.epochs,
        inner_steps=args.inner_steps,
        inner_lr=args.inner_lr,
        meta_lr=args.meta_lr,
        support_size=args.support_size,
        query_size=args.query_size,
        critic_cases=args.critic_cases,
        validation_interval=args.validation_interval,
        seed=args.seed,
        device=args.device,
        output_dir=args.output_dir,
    )


if "__file__" in globals() and __name__ == "__main__":
    main(parse_args())

## Notebook execution

Run this cell after generating `dnr_diagnostic.h5`. It uses the same `main()` implementation as the script, but passes a `RunConfig` object directly instead of parsing command-line arguments.

In [ ]:
# Example notebook run. Tune values here for experiments.
# results = main(RunConfig(
#     h5_path="dnr_diagnostic.h5",
#     epochs=50,
#     inner_steps=5,
#     critic_cases=16,
#     device="cuda",
#     output_dir="metaphygt_outputs",
# ))